In [ ]:
import numpy as np
import pandas as pd 

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
# ========================================
# CUSTOM CNN FOR CHEST X-RAY PNEUMONIA/ new
# ========================================

import os
import time
import copy
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    roc_curve,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report
)

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms

warnings.filterwarnings("ignore")

# =========================
# 1. Reproducibility + safe device setup
# =========================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

def get_safe_device():
    if torch.cuda.is_available():
        try:
            _ = torch.tensor([1.0]).to("cuda")
            torch.cuda.manual_seed_all(SEED)
            return torch.device("cuda")
        except Exception:
            pass
    return torch.device("cpu")

device = get_safe_device()
print("Using device:", device)

# =========================
# 2. Dataset path
# =========================
DATASET_ROOT = "/kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray/chest_xray"
TRAIN_DIR = os.path.join(DATASET_ROOT, "train")
VAL_DIR = os.path.join(DATASET_ROOT, "val")
TEST_DIR = os.path.join(DATASET_ROOT, "test")

IMG_EXTENSIONS = (".jpg", ".jpeg", ".png")